# 5.3 硬件感知优化

LLM推理的瓶颈取决于推理阶段：prefill阶段是计算受限（compute-bound），decode阶段是内存带宽受限（memory-bound）。硬件感知优化针对不同瓶颈采取不同策略。

## 什么是硬件感知优化？

硬件感知优化是在模型设计和部署时考虑目标硬件的特性，使模型与硬件协同达到最优性能。核心方法包括：
- **Roofline分析**：识别算子是计算受限还是带宽受限
- **量化感知架构**：设计对量化友好的模型结构
- **内存层次优化**：针对缓存层次优化数据访问模式

## 为什么需要硬件感知优化？

1. **LLM的双重瓶颈**：prefill阶段$O(n^2)$计算是compute-bound，decode阶段每步仅1个token是memory-bound
2. **端侧极端约束**：端侧设备的算力、带宽、功耗都远低于云端
3. **通用优化不够**：量化/剪枝等通用方法未考虑硬件特性，硬件感知优化可以进一步挖掘性能

## 硬件感知优化的数学原理

**Roofline模型**：算术强度$I = \frac{\text{FLOPs}}{\text{Bytes}}$决定性能上界：
$$P_{\max} = \min(I \cdot \text{BW}, \text{Peak FLOPS})$$
LLM decode阶段：$I_{\text{decode}} = \frac{2d}{d \cdot 2} = 1$（极低，memory-bound）
LLM prefill阶段：$I_{\text{prefill}} = \frac{2nd^2}{2nd + 2n^2d} \approx d$（较高，compute-bound）

**功耗模型**：$P = \alpha C V^2 f + V I_{\text{leak}}$，端侧需在功耗预算内最大化性能

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

---
## 5.3.1 Roofline模型深度分析

Roofline模型是硬件感知优化的核心分析工具，它将硬件性能特征和算子计算特征统一到一个可视化框架中。

### Roofline模型的数学基础

性能上界：
$$P_{\max} = \min(I \cdot \text{BW}, \text{Peak FLOPS})$$

拐点（ridge point）：$I_{\text{threshold}} = \frac{\text{Peak FLOPS}}{\text{BW}}$

- $I < I_{\text{threshold}}$：memory-bound，性能$= I \cdot \text{BW}$（线性增长）
- $I \geq I_{\text{threshold}}$：compute-bound，性能$= \text{Peak FLOPS}$（饱和）

### 不同硬件的Roofline特征

| 硬件 | Peak FLOPS | BW (GB/s) | 拐点 (FLOP/B) | 特征 |
|------|-----------|-----------|--------------|------|
| **NVIDIA A100** | 312 TFLOPS (FP16) | 2032 | 154 | 大多数LLM算子compute-bound |
| **NVIDIA RTX 4090** | 165 TFLOPS (FP16) | 1008 | 164 | 类似A100 |
| **高通Hexagon NPU** | 75 TOPS (INT8) | 60 | 1250 | 大多数LLM算子memory-bound |
| **Apple M4 ANE** | 38 TOPS (FP16) | 120 | 317 | 拐点较低，ANE带宽优势 |
| **ARM Cortex-A715** | 0.5 TFLOPS (FP16) | 40 | 12.5 | 几乎全部memory-bound |

**关键洞察**：端侧NPU/CPU的拐点远高于云端GPU，意味着在端侧更多的算子落在memory-bound区域，量化（减少数据搬运）的效果更显著。

### LLM关键算子的算术强度

| 算子 | FLOPs | Bytes (FP16) | 算术强度 (FLOP/B) | 瓶颈类型 |
|------|-------|-------------|-------------------|----------|
| **QKV投影 (prefill, B=4, L=512)** | $2 \times 4 \times 512 \times d^2 \times 3$ | $d^2 \times 3 \times 2 + 4 \times 512 \times d \times 2$ | $\approx d$ | compute-bound |
| **QKV投影 (decode, B=1, L=1)** | $2 \times 1 \times 1 \times d^2 \times 3$ | $d^2 \times 3 \times 2 + 1 \times 1 \times d \times 2$ | $\approx 1$ | memory-bound |
| **Attention Score (prefill)** | $2 \times B \times H \times L^2 \times d_h$ | $2 \times B \times L \times d$ | $\approx L$ | compute-bound (L大时) |
| **FFN (prefill)** | $2 \times B \times L \times d \times 4d$ | $d \times 4d \times 2$ | $\approx L$ | compute-bound |
| **FFN (decode)** | $2 \times 1 \times 1 \times d \times 4d$ | $d \times 4d \times 2$ | $\approx 1$ | memory-bound |

In [ ]:
class RooflineAnalyzer:
    """Roofline模型分析器 - 支持多硬件平台对比"""
    def __init__(self, peak_flops_tflops, peak_bandwidth_gbs, name='Hardware'):
        self.peak_flops = peak_flops_tflops * 1e12
        self.peak_bandwidth = peak_bandwidth_gbs * 1e9
        self.name = name
        self.ridge_point = self.peak_flops / self.peak_bandwidth

    def analyze_operation(self, name, flops, bytes_accessed):
        """分析单个操作的瓶颈"""
        arithmetic_intensity = flops / max(bytes_accessed, 1)
        is_compute_bound = arithmetic_intensity > self.ridge_point
        max_throughput = min(self.peak_flops, self.peak_bandwidth * arithmetic_intensity)
        efficiency = max_throughput / self.peak_flops if self.peak_flops > 0 else 0
        return {
            'name': name,
            'flops': flops,
            'bytes': bytes_accessed,
            'arithmetic_intensity': arithmetic_intensity,
            'is_compute_bound': is_compute_bound,
            'max_throughput_tflops': max_throughput / 1e12,
            'efficiency': efficiency,
        }

    def analyze_llm_layer(self, hidden_dim, n_layers, seq_len, n_heads,
                          batch_size=1, dtype_bytes=2, phase='both'):
        """分析LLM单层的计算特征"""
        head_dim = hidden_dim // n_heads
        results = {}

        if phase in ('prefill', 'both'):
            qkv_flops = 2 * batch_size * seq_len * hidden_dim * hidden_dim * 3
            qkv_bytes = (hidden_dim * hidden_dim * 3 * dtype_bytes +
                         batch_size * seq_len * hidden_dim * dtype_bytes)
            results['prefill_qkv'] = self.analyze_operation('Prefill QKV', qkv_flops, qkv_bytes)

            attn_flops = 2 * batch_size * n_heads * seq_len * seq_len * head_dim
            attn_bytes = 2 * batch_size * seq_len * hidden_dim * dtype_bytes
            results['prefill_attn'] = self.analyze_operation('Prefill Attention', attn_flops, attn_bytes)

            ffn_flops = 2 * batch_size * seq_len * hidden_dim * hidden_dim * 3
            ffn_bytes = hidden_dim * hidden_dim * 3 * dtype_bytes + batch_size * seq_len * hidden_dim * dtype_bytes
            results['prefill_ffn'] = self.analyze_operation('Prefill FFN', ffn_flops, ffn_bytes)

        if phase in ('decode', 'both'):
            qkv_flops = 2 * batch_size * 1 * hidden_dim * hidden_dim * 3
            qkv_bytes = (hidden_dim * hidden_dim * 3 * dtype_bytes +
                         batch_size * seq_len * hidden_dim * dtype_bytes * 2)
            results['decode_qkv'] = self.analyze_operation('Decode QKV', qkv_flops, qkv_bytes)

            ffn_flops = 2 * batch_size * 1 * hidden_dim * hidden_dim * 3
            ffn_bytes = hidden_dim * hidden_dim * 3 * dtype_bytes + batch_size * seq_len * hidden_dim * dtype_bytes * 2
            results['decode_ffn'] = self.analyze_operation('Decode FFN', ffn_flops, ffn_bytes)

        return results

hardware_platforms = [
    RooflineAnalyzer(75, 60, '高通Hexagon NPU'),
    RooflineAnalyzer(38, 120, 'Apple M4 ANE'),
    RooflineAnalyzer(0.5, 40, 'ARM Cortex-A715'),
    RooflineAnalyzer(165, 1008, 'NVIDIA RTX 4090'),
]

print("=== 不同硬件平台的Roofline特征 ===")
print(f"{'硬件':<22} {'峰值算力':<15} {'峰值带宽':<15} {'拐点(FLOP/B)':<15}")
print("-" * 67)
for hw in hardware_platforms:
    print(f"{hw.name:<22} {hw.peak_flops/1e12:.0f} TFLOPS     {hw.peak_bandwidth/1e9:.0f} GB/s       {hw.ridge_point:.0f}")

print(f"\n=== 7B模型在不同硬件上的瓶颈分析 (d=4096, L=512) ===")
for hw in hardware_platforms:
    results = hw.analyze_llm_layer(4096, 32, 512, 32, batch_size=1, dtype_bytes=2)
    print(f"\n--- {hw.name} (拐点={hw.ridge_point:.0f} FLOP/B) ---")
    for op_name, info in results.items():
        bottleneck = '计算受限' if info['is_compute_bound'] else '带宽受限'
        print(f"  {op_name:<18}: AI={info['arithmetic_intensity']:.1f} FLOP/B, {bottleneck}, "
              f"效率={info['efficiency']:.1%}")

print(f"\n=== 关键洞察 ===")
print(f"1. 端侧NPU/CPU的拐点远高于GPU → 更多算子落在带宽受限区域")
print(f"2. Decode阶段所有硬件都是带宽受限 → 量化减少搬运量最有效")
print(f"3. Prefill阶段在端侧NPU也可能带宽受限 → 需要算子融合减少访存")
print(f"4. 量化对带宽受限算子加速=精度缩减比 (INT4→4x, INT8→2x)")
print(f"5. 量化对计算受限算子加速有限 → 需要硬件原生低精度MAC支持")

---
## 5.3.2 内存层次优化

理解硬件的内存层次结构，并针对其优化数据访问模式，是硬件感知优化的核心实践。

### 典型端侧NPU内存层次

```
┌─────────────────────────────────────────────────┐
│                   DRAM                           │
│  容量: 4-16GB    带宽: 30-120 GB/s   延迟: ~100ns │
│  存储: 模型权重 / KV Cache / 完整激活值           │
└──────────────────────┬──────────────────────────┘
                       │ DMA (异步, 多通道)
┌──────────────────────▼──────────────────────────┐
│              片上SRAM / 共享缓存                   │
│  容量: 256KB-8MB  带宽: 0.8-2 TB/s   延迟: ~1-5ns │
│  存储: 当前tile权重 / 当前层激活值 / KV tile       │
└──────────────────────┬──────────────────────────┘
                       │ 寄存器文件
┌──────────────────────▼──────────────────────────┐
│               MAC阵列寄存器                       │
│  容量: 4-32KB    带宽: >10 TB/s     延迟: <1ns    │
│  存储: 当前计算的微tile                            │
└─────────────────────────────────────────────────┘
```

### Tiling策略：SRAM容量约束下的算子分解

Tiling是将大算子按SRAM容量切分为小tile，每次只处理一个tile的数据。核心原则：

**最大化数据在SRAM中的复用次数，最小化DRAM访问次数**

以矩阵乘$C = AB$为例（$A \in \mathbb{R}^{M \times K}$, $B \in \mathbb{R}^{K \times N}$）：

- **无Tiling**：需从DRAM读取$A$和$B$各$M$次和$N$次
- **Tiling**（tile大小$T_M \times T_K$和$T_K \times T_N$）：
  - 每个tile的$A$块在SRAM中被复用$N/T_N$次
  - 每个tile的$B$块在SRAM中被复用$M/T_M$次
  - DRAM访问量从$O(MNK)$降至$O(MNK/T)$，其中$T$为复用因子

### Flash Attention的Tiling原理

Flash Attention是Tiling策略在Attention算子上的经典应用：

$$\text{softmax}(QK^T)V = \frac{\sum_j e^{q_i \cdot k_j} v_j}{\sum_j e^{q_i \cdot k_j}}$$

分块计算的关键：维护running sum和running max，避免完整$QK^T$矩阵：
- $m_i^{(j)} = \max(m_i^{(j-1)}, \max(Q_i K_j^T))$（running max）
- $l_i^{(j)} = e^{m_i^{(j-1)} - m_i^{(j)}} l_i^{(j-1)} + \text{rowsum}(e^{Q_i K_j^T - m_i^{(j)}})$（running sum）
- $O_i = \text{diag}(e^{m_i^{(j-1)} - m_i^{(j)}}) O_i^{(j-1)} + e^{Q_i K_j^T - m_i^{(j)}} V_j$（running output）

In [ ]:
class TilingOptimizer:
    """Tiling策略优化器 - 在SRAM约束下搜索最优tile大小"""
    def __init__(self, sram_capacity_kb=4096, dtype_bytes=2):
        self.sram = sram_capacity_kb * 1024
        self.dtype = dtype_bytes

    def analyze_gemm_tiling(self, M, N, K):
        """分析GEMM的Tiling策略"""
        total_sram_needed = (M * K + K * N + M * N) * self.dtype
        no_tiling_fits = total_sram_needed <= self.sram

        tile_configs = []
        for tm in [256, 128, 64, 32]:
            for tn in [256, 128, 64, 32]:
                for tk in [128, 64, 32]:
                    tile_sram = (tm * tk + tk * tn + tm * tn) * self.dtype
                    if tile_sram <= self.sram:
                        reuse_a = N / tn
                        reuse_b = M / tm
                        n_tiles = (M // tm) * (N // tn) * (K // tk)
                        dram_reads = (M * K + K * N) * self.dtype * (K // tk)
                        tile_configs.append({
                            'Tm': tm, 'Tn': tn, 'Tk': tk,
                            'sram_usage_kb': tile_sram / 1024,
                            'reuse_factor': reuse_a * reuse_b,
                            'n_tiles': n_tiles,
                        })

        tile_configs.sort(key=lambda x: -x['reuse_factor'])
        return {
            'no_tiling_fits': no_tiling_fits,
            'total_sram_mb': total_sram_needed / 1024 / 1024,
            'best_tile': tile_configs[0] if tile_configs else None,
        }

    def analyze_flash_attention_tiling(self, batch, seq_len, hidden_dim, n_heads):
        """分析Flash Attention的Tiling策略"""
        head_dim = hidden_dim // n_heads
        full_attn_sram = batch * n_heads * seq_len * seq_len * self.dtype
        full_sram = full_attn_sram + 2 * batch * seq_len * hidden_dim * self.dtype

        tile_configs = []
        for bq in [64, 32, 16]:
            for bk in [128, 64, 32]:
                tile_sram = (bq * head_dim + bk * head_dim + bq * bk + bq * head_dim) * n_heads * batch * self.dtype
                if tile_sram <= self.sram:
                    tile_configs.append({
                        'Bq': bq, 'Bk': bk,
                        'sram_usage_kb': tile_sram / 1024,
                        'n_q_tiles': (seq_len + bq - 1) // bq,
                        'n_k_tiles': (seq_len + bk - 1) // bk,
                    })

        tile_configs.sort(key=lambda x: -x['Bq'] * x['Bk'])
        return {
            'full_sram_mb': full_sram / 1024 / 1024,
            'fits_sram': full_sram <= self.sram,
            'best_tile': tile_configs[0] if tile_configs else None,
        }

optimizer = TilingOptimizer(sram_capacity_kb=4096, dtype_bytes=2)

print("=== GEMM Tiling分析 (7B模型FFN层, M=1, N=4096, K=4096) ===")
result = optimizer.analyze_gemm_tiling(M=1, N=4096, K=4096)
print(f"无Tiling SRAM需求: {result['total_sram_mb']:.2f} MB")
print(f"是否可fit SRAM: {'是' if result['no_tiling_fits'] else '否'}")
if result['best_tile']:
    t = result['best_tile']
    print(f"最优Tiling: Tm={t['Tm']}, Tn={t['Tn']}, Tk={t['Tk']}")
    print(f"  SRAM使用: {t['sram_usage_kb']:.0f} KB, 复用因子: {t['reuse_factor']:.0f}")

print(f"\n=== Flash Attention Tiling分析 (7B, seq=2048) ===")
fa_result = optimizer.analyze_flash_attention_tiling(1, 2048, 4096, 32)
print(f"完整Attention SRAM需求: {fa_result['full_sram_mb']:.2f} MB")
print(f"是否可fit SRAM: {'是' if fa_result['fits_sram'] else '否'}")
if fa_result['best_tile']:
    t = fa_result['best_tile']
    print(f"最优Tiling: Bq={t['Bq']}, Bk={t['Bk']}")
    print(f"  SRAM使用: {t['sram_usage_kb']:.0f} KB")
    print(f"  Q tiles: {t['n_q_tiles']}, K tiles: {t['n_k_tiles']}")

print(f"\n=== Tiling优化的关键原则 ===")
print(f"1. SRAM容量是硬约束: tile大小必须fit进SRAM")
print(f"2. 最大化复用: tile内的数据应被尽可能多次计算复用")
print(f"3. 最小化DRAM访问: 每个数据块只从DRAM读取一次，在SRAM中多次使用")
print(f"4. Flash Attention: 通过分块+running统计避免O(L²)的SRAM需求")
print(f"5. 实际NPU SDK自动执行Tiling，但理解原理有助于调优和调试")

---
## 5.3.3 双缓冲与流水线优化

双缓冲（Double Buffering）和流水线（Pipelining）是隐藏数据搬运延迟的关键技术。

### 双缓冲原理

```
无双缓冲:
时间 →  T1    T2    T3    T4    T5    T6
DMA   [搬运W1]------[搬运W2]------[搬运W3]------
NPU          [计算W1]------[计算W2]------[计算W3]--

双缓冲:
时间 →  T1    T2    T3    T4    T5
DMA   [搬运W1][搬运W2][搬运W3][搬运W4]------
NPU          [计算W1][计算W2][计算W3][计算W4]
              ↑ W2在NPU计算W1时已预取到SRAM
```

双缓冲将SRAM分为两个buffer：
- **Buffer A**：NPU正在计算的数据
- **Buffer B**：DMA正在预取的下一层数据

当NPU完成Buffer A的计算后，交换A和B的指针，NPU立即开始计算Buffer B，同时DMA开始预取下一层到Buffer A。

### 流水线并行

将多层Transformer的推理组织为流水线：

```
时间 →  T1    T2    T3    T4    T5    T6    T7
Layer0 [L0计算]------[L0计算]------
Layer1       [L1计算]------[L1计算]------
Layer2             [L2计算]------[L2计算]------
DMA    [预取L0][预取L1][预取L2][预取L0][预取L1][预取L2]
```

### 延迟隐藏的条件

双缓冲有效的条件：$T_{\text{DMA}} \leq T_{\text{compute}}$

即数据搬运时间不超过计算时间。对于decode阶段（memory-bound），$T_{\text{compute}} < T_{\text{DMA}}$，双缓冲无法完全隐藏延迟，但可以部分重叠。

In [ ]:
class DoubleBufferSimulator:
    """双缓冲与流水线模拟器"""
    def __init__(self, sram_kb=4096, dram_bw_gbs=60, npu_tops=75, dtype_bytes=2):
        self.sram = sram_kb * 1024
        self.dram_bw = dram_bw_gbs * 1e9
        self.npu_flops = npu_tops * 1e12
        self.dtype = dtype_bytes

    def simulate_layer_inference(self, hidden_dim, is_decode=True):
        """模拟单层推理的时间分解"""
        weight_bytes = hidden_dim * hidden_dim * 7 * self.dtype
        if is_decode:
            compute_flops = 2 * 1 * 1 * hidden_dim * hidden_dim * 7
        else:
            compute_flops = 2 * 1 * 512 * hidden_dim * hidden_dim * 7

        dma_time_us = weight_bytes / self.dram_bw * 1e6
        compute_time_us = compute_flops / self.npu_flops * 1e6

        no_overlap = dma_time_us + compute_time_us
        with_overlap = max(dma_time_us, compute_time_us)
        speedup = no_overlap / with_overlap if with_overlap > 0 else 1.0

        return {
            'dma_us': dma_time_us,
            'compute_us': compute_time_us,
            'no_overlap_us': no_overlap,
            'with_overlap_us': with_overlap,
            'speedup': speedup,
            'bottleneck': 'DMA' if dma_time_us > compute_time_us else 'Compute',
        }

    def simulate_pipeline(self, n_layers, hidden_dim, is_decode=True):
        """模拟多层流水线推理"""
        layer_result = self.simulate_layer_inference(hidden_dim, is_decode)
        no_pipeline_total = layer_result['no_overlap_us'] * n_layers
        pipeline_total = layer_result['with_overlap_us'] * n_layers
        return {
            'per_layer': layer_result,
            'total_no_pipeline_us': no_pipeline_total,
            'total_pipeline_us': pipeline_total,
            'pipeline_speedup': no_pipeline_total / pipeline_total if pipeline_total > 0 else 1.0,
        }

sim = DoubleBufferSimulator(sram_kb=4096, dram_bw_gbs=60, npu_tops=75)

print("=== 双缓冲优化效果分析 (7B模型, 高通Hexagon) ===")

for phase, is_decode in [('Prefill', False), ('Decode', True)]:
    result = sim.simulate_pipeline(32, 4096, is_decode)
    layer = result['per_layer']
    print(f"\n--- {phase}阶段 ---")
    print(f"  单层DMA时间: {layer['dma_us']:.1f} μs")
    print(f"  单层计算时间: {layer['compute_us']:.1f} μs")
    print(f"  瓶颈: {layer['bottleneck']}")
    print(f"  无重叠总延迟: {result['total_no_pipeline_us']/1000:.2f} ms")
    print(f"  双缓冲总延迟: {result['total_pipeline_us']/1000:.2f} ms")
    print(f"  加速比: {result['pipeline_speedup']:.2f}x")

print(f"\n=== 不同优化策略的叠加效果 ===")
strategies = [
    ('基线(FP16, 无优化)', 2, False, 1.0),
    ('INT4量化', 0.5, False, 1.0),
    ('INT4 + 双缓冲', 0.5, True, 1.0),
    ('INT4 + 双缓冲 + 算子融合', 0.5, True, 0.85),
]

print(f"\n{'策略':<30} {'Decode延迟(ms)':<18} {'vs基线':<10}")
print("-" * 58)
base_latency = None
for name, dtype_factor, overlap, fusion_factor in strategies:
    sim_temp = DoubleBufferSimulator(sram_kb=4096, dram_bw_gbs=60, npu_tops=75, dtype_bytes=dtype_factor)
    result = sim_temp.simulate_pipeline(32, 4096, is_decode=True)
    latency = result['total_pipeline_us' if overlap else 'total_no_pipeline_us'] / 1000 * fusion_factor
    if base_latency is None:
        base_latency = latency
    print(f"{name:<30} {latency:<18.2f} {latency/base_latency:.2f}x")

print(f"\n=== 产业实践要点 ===")
print(f"1. 量化是第一优化: INT4将数据搬运量减少4x，直接加速decode")
print(f"2. 双缓冲在compute-bound场景效果最好，memory-bound场景效果有限")
print(f"3. 算子融合减少中间结果的DRAM写回，进一步降低带宽需求")
print(f"4. 三者叠加可实现约8-10x的decode加速")
print(f"5. 实际NPU SDK通常内置双缓冲和算子融合，但需确认已启用")

---
## 5.3.4 量化对硬件性能的影响分析

量化不仅减少模型体积，更重要的是改变算子的算术强度，从而改变其在Roofline上的位置。

### 量化如何改变算术强度

对于矩阵乘$Y = XW$：
- **FP16**：$I = \frac{2MNK}{2MN + 2NK + 2MK} \approx \frac{2MNK}{2NK} = M$（权重为主）
- **INT4权重**：$I = \frac{2MNK}{2MN + 0.5NK + 2MK} \approx \frac{2MNK}{0.5NK} = 4M$

INT4量化将权重的数据搬运量减少4x，算术强度提升约4x，使算子向Roofline的compute-bound区域移动。

### 量化对不同瓶颈的加速效果

| 场景 | 瓶颈 | INT4加速 | 原因 |
|------|------|----------|------|
| **Decode (batch=1)** | 带宽受限 | ~3-4x | 权重搬运量减少4x，直接加速 |
| **Prefill (batch=4, L=512)** | 计算受限 | ~1.0-1.5x | 瓶颈在计算而非搬运 |
| **Prefill (batch=1, L=128)** | 混合 | ~2x | 部分算子带宽受限 |
| **KV Cache** | 带宽受限 | ~2-4x | KV量化直接减少读取量 |

### 混合精度对硬件利用率的影响

| 配置 | 权重精度 | 激活精度 | MAC利用率 | 带宽需求 | 适用场景 |
|------|---------|---------|----------|---------|----------|
| W16A16 | FP16 | FP16 | 100% (FP16 MAC) | 最高 | 精度优先 |
| W8A16 | INT8 | FP16 | 200% (INT8 MAC) | 50% | 平衡精度与速度 |
| W4A16 | INT4 | FP16 | 400% (INT4 MAC) | 25% | 速度优先 |
| W8A8 | INT8 | INT8 | 200% | 50% | NPU友好 |
| W4A8 | INT4 | INT8 | 400% | 25% | NPU最优 |

In [ ]:
class QuantizationImpactAnalyzer:
    """量化对硬件性能影响的分析器"""
    def __init__(self, peak_flops_tflops, peak_bw_gbs, name='Hardware'):
        self.peak_flops = peak_flops_tflops * 1e12
        self.peak_bw = peak_bw_gbs * 1e9
        self.name = name

    def analyze_decode_performance(self, hidden_dim, n_layers, n_heads,
                                   weight_dtype_bytes, kv_dtype_bytes=2,
                                   batch_size=1, seq_len=512):
        """分析不同量化配置下的decode性能"""
        weight_bytes = n_layers * hidden_dim * hidden_dim * 7 * weight_dtype_bytes
        kv_bytes = 2 * n_layers * batch_size * seq_len * hidden_dim * kv_dtype_bytes
        total_bytes = weight_bytes + kv_bytes

        compute_flops = 2 * batch_size * 1 * hidden_dim * hidden_dim * 7 * n_layers
        mac_multiplier = 2.0 / weight_dtype_bytes
        effective_flops = compute_flops / mac_multiplier

        bandwidth_limited_time = total_bytes / self.peak_bw
        compute_limited_time = effective_flops / self.peak_flops
        actual_time = max(bandwidth_limited_time, compute_limited_time)

        return {
            'weight_bytes_mb': weight_bytes / 1e6,
            'kv_bytes_mb': kv_bytes / 1e6,
            'total_bytes_mb': total_bytes / 1e6,
            'bandwidth_limited_ms': bandwidth_limited_time * 1000,
            'compute_limited_ms': compute_limited_time * 1000,
            'actual_ms': actual_time * 1000,
            'bottleneck': 'bandwidth' if bandwidth_limited_time > compute_limited_time else 'compute',
            'tokens_per_second': 1.0 / actual_time if actual_time > 0 else 0,
        }

analyzer = QuantizationImpactAnalyzer(75, 60, '高通Hexagon NPU')

configs = [
    ('W16A16 (FP16)', 2, 2),
    ('W8A16', 1, 2),
    ('W4A16', 0.5, 2),
    ('W4A16 + KV INT8', 0.5, 1),
    ('W4A8 + KV INT4', 0.5, 0.5),
]

print(f"=== 量化配置对Decode性能的影响 (7B模型, {analyzer.name}) ===")
print(f"假设: batch=1, seq_len=512")
print(f"\n{'配置':<22} {'权重(MB)':<10} {'KV(MB)':<10} {'总计(MB)':<10} {'延迟(ms)':<12} {'tok/s':<10} {'瓶颈':<10}")
print("-" * 84)

base_tps = None
for name, w_bytes, kv_bytes in configs:
    result = analyzer.analyze_decode_performance(4096, 32, 32, w_bytes, kv_bytes)
    if base_tps is None:
        base_tps = result['tokens_per_second']
    speedup = result['tokens_per_second'] / base_tps
    print(f"{name:<22} {result['weight_bytes_mb']:<10.0f} {result['kv_bytes_mb']:<10.0f} "
          f"{result['total_bytes_mb']:<10.0f} {result['actual_ms']:<12.2f} "
          f"{result['tokens_per_second']:<10.1f} {result['bottleneck']:<10}")

print(f"\n=== 量化加速比 ===")
for name, w_bytes, kv_bytes in configs:
    result = analyzer.analyze_decode_performance(4096, 32, 32, w_bytes, kv_bytes)
    speedup = result['tokens_per_second'] / base_tps
    print(f"  {name}: {speedup:.2f}x vs FP16")

print(f"\n=== 关键洞察 ===")
print(f"1. W4A16相比W16A16, decode加速约3-4x (权重搬运减少4x)")
print(f"2. KV量化额外加速约1.2-1.5x (KV占decode总搬运量的20-40%)")
print(f"3. W4A8是NPU最优配置: INT4权重+INT8激活, 充分利用NPU MAC阵列")
print(f"4. 量化后瓶颈可能从带宽转向计算, 需重新评估优化方向")
print(f"5. 实际加速还受NPU量化kernel效率影响, 理论值是上界")

---
## 5.3.5 功耗预算优化

端侧设备有严格的功耗约束，持续推理会导致芯片过热降频，性能急剧下降。

### 功耗模型

芯片动态功耗：
$$P_{\text{dynamic}} = \alpha \cdot C \cdot V^2 \cdot f$$

- $\alpha$：活动因子（0-1），表示电路翻转频率
- $C$：电容，由工艺节点决定
- $V$：供电电压
- $f$：时钟频率

漏电功耗：
$$P_{\text{leak}} = V \cdot I_{\text{leak}}$$

总功耗：$P = P_{\text{dynamic}} + P_{\text{leak}}$

### 热节流机制

```
温度 → 超过T_throttle → 降频(f↓, V↓) → 性能下降 → 温度降低 → 恢复频率
        ↑                                                    │
        └────────────────────────────────────────────────────┘
                         热节流循环
```

| 设备 | TDP | 典型降频阈值 | 降频后性能 |
|------|-----|------------|----------|
| 手机NPU | 3-5W | ~45°C | 降至50-70%性能 |
| 平板NPU | 5-8W | ~50°C | 降至60-80%性能 |
| 笔记本CPU | 15-45W | ~95°C | 降至70-90%性能 |

### 功耗优化策略

1. **DVFS（动态电压频率调整）**：降低$f$和$V$，功耗按$V^2 f$降低
   - 适用：后台推理、低优先级任务
   - 效果：20-40%功耗降低，10-30%性能损失

2. **精度自适应**：根据热状态动态切换计算精度
   - 温度低 → W4A16（高精度）
   - 温度高 → W4A8（低精度，NPU更高效）
   - 效果：30-50%功耗降低

3. **推理调度**：错峰执行推理任务
   - 交互式请求：立即执行，用NPU
   - 后台请求：延迟执行，用CPU低频
   - 效果：15-25%功耗降低

4. **模型切换**：根据热状态切换模型大小
   - 温度低 → 7B模型（高质量）
   - 温度高 → 1.5B模型（低功耗）
   - 效果：40-60%功耗降低

In [ ]:
class PowerBudgetOptimizer:
    """功耗预算优化器 - 在TDP约束下最大化推理性能"""
    def __init__(self, tdp_watts=5.0, throttle_temp_c=45):
        self.tdp = tdp_watts
        self.throttle_temp = throttle_temp_c

    def estimate_power(self, model_params_b, quant_dtype_bytes, npu_utilization=0.8):
        """估算推理功耗"""
        base_power = self.tdp * npu_utilization
        quant_factor = quant_dtype_bytes / 2.0
        estimated_power = base_power * (0.5 + 0.5 * quant_factor)
        return estimated_power

    def estimate_thermal_behavior(self, power_watts, duration_s, ambient_temp=25):
        """估算热行为 (简化模型)"""
        thermal_resistance = 15
        steady_state_temp = ambient_temp + power_watts * thermal_resistance
        time_constant = 30
        temp_at_t = ambient_temp + (steady_state_temp - ambient_temp) * (1 - np.exp(-duration_s / time_constant))
        will_throttle = temp_at_t >= self.throttle_temp
        time_to_throttle = None
        if steady_state_temp >= self.throttle_temp:
            time_to_throttle = -time_constant * np.log(
                (self.throttle_temp - ambient_temp) / (steady_state_temp - ambient_temp))
        return {
            'steady_state_temp': steady_state_temp,
            'temp_at_t': temp_at_t,
            'will_throttle': will_throttle,
            'time_to_throttle_s': time_to_throttle,
        }

    def recommend_strategy(self, scenario, current_temp=30):
        """根据场景和温度推荐功耗优化策略"""
        strategies = {
            'realtime_chat': {
                'normal': ('W4A16, NPU全速', '最高质量'),
                'warm': ('W4A16, DVFS降频10%', '轻微延迟增加'),
                'hot': ('W4A8, DVFS降频20%', '精度略降, 延迟增加'),
                'critical': ('切换1.5B模型', '质量下降, 避免降频'),
            },
            'background': {
                'normal': ('W4A8, CPU低频推理', '最低功耗'),
                'warm': ('W4A8, 延迟推理', '错峰执行'),
                'hot': ('暂停推理', '等待降温'),
                'critical': ('暂停推理', '等待降温'),
            },
        }

        if current_temp < 35:
            thermal_state = 'normal'
        elif current_temp < 40:
            thermal_state = 'warm'
        elif current_temp < self.throttle_temp:
            thermal_state = 'hot'
        else:
            thermal_state = 'critical'

        scenario_strategies = strategies.get(scenario, strategies['realtime_chat'])
        return thermal_state, scenario_strategies.get(thermal_state, ('N/A', 'N/A'))

optimizer = PowerBudgetOptimizer(tdp_watts=5.0, throttle_temp_c=45)

print(f"=== 功耗与热行为分析 (TDP={optimizer.tdp}W) ===")

for model_name, params_b, dtype_bytes in [('7B W4A16', 7, 0.5), ('7B W4A8', 7, 0.5), ('1.5B W4A16', 1.5, 0.5)]:
    power = optimizer.estimate_power(params_b, dtype_bytes)
    thermal = optimizer.estimate_thermal_behavior(power, duration_s=60)
    print(f"\n{model_name}:")
    print(f"  估算功耗: {power:.1f}W")
    print(f"  稳态温度: {thermal['steady_state_temp']:.1f}°C")
    print(f"  60秒后温度: {thermal['temp_at_t']:.1f}°C")
    if thermal['time_to_throttle_s']:
        print(f"  预计{thermal['time_to_throttle_s']:.0f}秒后触发降频")
    else:
        print(f"  不会触发降频")

print(f"\n=== 自适应功耗策略推荐 ===")
for scenario in ['realtime_chat', 'background']:
    print(f"\n--- {scenario} ---")
    for temp in [30, 38, 43, 46]:
        state, (action, effect) = optimizer.recommend_strategy(scenario, temp)
        print(f"  {temp}°C ({state}): {action} → {effect}")

print(f"\n=== 产业级功耗管理要点 ===")
print(f"1. 热节流是端侧推理的最大敌人: 降频后性能可能降至50%")
print(f"2. 温度监控是基础: 实时读取芯片温度传感器，动态调整策略")
print(f"3. 分级降级: 从降频→降精度→换小模型→暂停，逐步降级")
print(f"4. 预判式调度: 根据温度趋势提前降级，避免突然降频")
print(f"5. 间歇推理: 长对话中插入冷却间隔，避免持续高负载")
print(f"6. NPU比CPU/GPU能效比高3-10x: 尽量将计算卸载到NPU")

---
## 5.3.6 硬件感知模型设计

硬件感知优化不仅在部署阶段进行，还可以在模型设计阶段就考虑硬件特性，设计对硬件友好的模型架构。

### 量化友好的架构设计

| 设计选择 | 量化友好 | 量化不友好 | 原因 |
|---------|---------|-----------|------|
| **激活函数** | ReLU/GELU | SiGLU/SwiGLU | SiGLU有乘法门控，量化误差放大 |
| **归一化** | BatchNorm | LayerNorm/RMSNorm | BN可融合到Conv，LN需逐token计算 |
| **注意力** | GQA/MQA | MHA | GQA减少KV头数，量化收益更大 |
| **FFN比例** | 2d (标准) | 8d/3 (MoE) | 标准FFN权重均匀，MoE权重稀疏 |
| **嵌入** | 共享嵌入 | 独立lm_head | 共享嵌入量化一次，独立需两次 |

### NPU友好的架构设计

| 设计选择 | NPU友好 | NPU不友好 | 原因 |
|---------|---------|-----------|------|
| **hidden_dim** | 1024/2048/4096 (2的幂) | 1536/3072 | NPU MAC阵列对2的幂对齐更高效 |
| **注意力** | GQA (8组) | MHA (32头) | GQA减少KV数量，NPU内存压力小 |
| **序列长度** | 固定长度/padding | 动态长度 | NPU编译期需固定shape |
| **FFN** | 标准FFN | SwiGLU | SwiGLU需逐元素乘，NPU额外开销 |
| **位置编码** | ALiBi / 无RoPE | RoPE | RoPE需逐元素乘加，NPU额外开销 |

### 硬件感知NAS（Neural Architecture Search）

在搜索空间中加入硬件约束：

$$\min_{\theta} \mathcal{L}(\theta) \quad \text{s.t.} \quad \text{Latency}(\theta, H) \leq T_{\text{budget}} \quad \text{and} \quad \text{Memory}(\theta) \leq M_{\text{budget}}$$

其中$H$为目标硬件平台，$T_{\text{budget}}$和$M_{\text{budget}}$分别为延迟和内存预算。

In [ ]:
class HardwareAwareModelEvaluator:
    """硬件感知模型评估器"""
    def __init__(self, target_hardware='npu'):
        self.target = target_hardware

    def evaluate_config(self, config_name, hidden_dim, n_heads, n_kv_heads,
                        ffn_mult, use_swiglu, use_rope, n_layers):
        """评估模型配置的硬件友好度"""
        scores = {}

        is_power_of_2 = (hidden_dim & (hidden_dim - 1)) == 0
        scores['dim_alignment'] = 1.0 if is_power_of_2 else 0.7

        kv_ratio = n_kv_heads / n_heads
        scores['kv_efficiency'] = kv_ratio

        scores['quantization_friendliness'] = 0.7 if use_swiglu else 0.9

        scores['npu_operator_efficiency'] = 0.7 if use_rope else 0.9

        total_params_b = n_layers * (
            hidden_dim * hidden_dim * (1 + 2 * kv_ratio + ffn_mult) * (2 if use_swiglu else 1)
        ) / 1e9
        scores['model_size_gb'] = total_params_b * 0.5
        scores['fits_mobile'] = 1.0 if scores['model_size_gb'] < 4 else 0.5 if scores['model_size_gb'] < 8 else 0.0

        overall = (scores['dim_alignment'] * 0.15 +
                   scores['kv_efficiency'] * 0.25 +
                   scores['quantization_friendliness'] * 0.25 +
                   scores['npu_operator_efficiency'] * 0.15 +
                   scores['fits_mobile'] * 0.20)

        return {
            'name': config_name,
            'scores': scores,
            'overall_score': overall,
            'model_size_gb': scores['model_size_gb'],
        }

evaluator = HardwareAwareModelEvaluator('npu')

configs = [
    ('Llama-2-7B (MHA)', 4096, 32, 32, 2.7, True, True, 32),
    ('Llama-2-70B (GQA)', 8192, 64, 8, 2.7, True, True, 80),
    ('Phi-3-mini (GQA)', 3072, 32, 8, 2.7, True, True, 32),
    ('Qwen2.5-1.5B (GQA)', 1536, 12, 2, 2.7, True, True, 28),
    ('MobileLLM (GQA, 无RoPE)', 2048, 16, 4, 2.0, False, False, 24),
]

print("=== 硬件感知模型架构评估 ===")
print(f"\n{'模型':<25} {'维度对齐':<10} {'KV效率':<10} {'量化友好':<10} {'NPU效率':<10} {'移动端适配':<10} {'综合':<8} {'大小(GB)':<10}")
print("-" * 93)

for cfg in configs:
    result = evaluator.evaluate_config(*cfg)
    s = result['scores']
    print(f"{result['name']:<25} {s['dim_alignment']:<10.2f} {s['kv_efficiency']:<10.2f} "
          f"{s['quantization_friendliness']:<10.2f} {s['npu_operator_efficiency']:<10.2f} "
          f"{s['fits_mobile']:<10.2f} {result['overall_score']:<8.2f} {result['model_size_gb']:<10.1f}")

print(f"\n=== 硬件感知设计建议 ===")
print(f"1. GQA是端侧必备: KV头数减少4-8x, 直接降低内存和带宽需求")
print(f"2. hidden_dim对齐2的幂: NPU MAC阵列效率更高")
print(f"3. 小模型(1.5B-3B)是端侧甜蜜点: 4GB内存预算内可INT4量化部署")
print(f"4. SwiGLU虽精度好但量化不友好: 端侧可考虑GELU替代")
print(f"5. RoPE增加NPU开销: 端侧可考虑ALiBi或无位置编码方案")
print(f"6. 硬件感知NAS: 在搜索空间中加入延迟和内存约束")

---
## 5.3.7 权重预取与流式加载

端侧设备的内存有限，无法一次性加载所有模型权重。权重预取（Weight Prefetching）和流式加载（Weight Streaming）是解决这一矛盾的关键技术。

### 权重预取策略

权重预取利用NPU计算当前层时，DMA异步预取下一层的权重到SRAM：

$$T_{\text{layer}} = \max(T_{\text{compute}}^{(i)}, T_{\text{prefetch}}^{(i+1)})$$

当$T_{\text{compute}}^{(i)} \geq T_{\text{prefetch}}^{(i+1)}$时，预取完全隐藏了权重加载延迟。

### 权重流式加载

当模型权重无法全部fit进DRAM时，需要按层流式加载：

```
DRAM (4GB)                    Flash (256GB)
┌──────────────┐              ┌──────────────────┐
│ Layer 0-15   │ ← 按需加载 ← │ 完整模型权重      │
│ (当前半段)    │              │ (70B, ~35GB)     │
└──────────────┘              └──────────────────┘
     ↓ 计算完Layer 0-15后
┌──────────────┐
│ Layer 16-31  │ ← 加载后半段
└──────────────┘
```

### 预取 vs 流式加载 vs 全量加载

| 策略 | 内存需求 | 延迟开销 | 适用场景 |
|------|---------|---------|----------|
| **全量加载** | 模型全部权重 | 无额外延迟 | 内存充足(>模型大小) |
| **权重预取** | 2层权重(SRAM) | 可忽略(计算重叠) | NPU推理标准方案 |
| **流式加载** | 部分层权重(DRAM) | Flash读取延迟 | 内存不足(大模型) |
| **混合策略** | 热层常驻+冷层流式 | 热层无延迟 | 部分层高频访问 |

### 产业实践：llama.cpp的权重加载策略

llama.cpp使用mmap实现零拷贝权重加载：
1. **mmap映射**：将GGUF文件映射到虚拟地址空间，OS按需分页加载
2. **预取提示**：madvise(MADV_WILLNEED)提示OS预取下一层权重
3. **大页支持**：使用Huge Pages减少TLB miss，提升权重读取速度
4. **NUMA感知**：在多NUMA节点系统上，权重分配到推理线程所在节点

In [ ]:
class WeightPrefetchSimulator:
    """权重预取与流式加载模拟器"""
    def __init__(self, dram_bw_gbs=60, flash_bw_gbs=2, npu_tops=75, sram_kb=4096):
        self.dram_bw = dram_bw_gbs * 1e9
        self.flash_bw = flash_bw_gbs * 1e9
        self.npu_flops = npu_tops * 1e12
        self.sram = sram_kb * 1024

    def simulate_prefetch(self, hidden_dim, n_layers, dtype_bytes=0.5):
        """模拟权重预取策略"""
        layer_weight_bytes = hidden_dim * hidden_dim * 7 * dtype_bytes
        layer_compute_flops = 2 * 1 * 1 * hidden_dim * hidden_dim * 7

        prefetch_time_us = layer_weight_bytes / self.dram_bw * 1e6
        compute_time_us = layer_compute_flops / self.npu_flops * 1e6

        no_prefetch_total = (prefetch_time_us + compute_time_us) * n_layers
        prefetch_total = max(prefetch_time_us, compute_time_us) * n_layers + min(prefetch_time_us, compute_time_us)
        speedup = no_prefetch_total / prefetch_total if prefetch_total > 0 else 1.0

        return {
            'layer_weight_mb': layer_weight_bytes / 1e6,
            'prefetch_time_us': prefetch_time_us,
            'compute_time_us': compute_time_us,
            'no_prefetch_total_ms': no_prefetch_total / 1000,
            'prefetch_total_ms': prefetch_total / 1000,
            'speedup': speedup,
            'prefetch_fully_hidden': compute_time_us >= prefetch_time_us,
        }

    def simulate_streaming(self, hidden_dim, n_layers, available_dram_mb, dtype_bytes=0.5):
        """模拟流式加载策略"""
        total_weight_mb = n_layers * hidden_dim * hidden_dim * 7 * dtype_bytes / 1e6
        layers_fit = int(available_dram_mb / (hidden_dim * hidden_dim * 7 * dtype_bytes / 1e6))
        n_chunks = (n_layers + layers_fit - 1) // layers_fit

        chunk_weight_mb = layers_fit * hidden_dim * hidden_dim * 7 * dtype_bytes / 1e6
        flash_load_time_ms = chunk_weight_mb / (self.flash_bw / 1e6 * 1000)

        return {
            'total_weight_mb': total_weight_mb,
            'available_dram_mb': available_dram_mb,
            'layers_fit': layers_fit,
            'n_chunks': n_chunks,
            'flash_load_time_per_chunk_ms': flash_load_time_ms,
            'total_flash_load_ms': flash_load_time_ms * n_chunks,
        }

sim = WeightPrefetchSimulator()
prefetch_result = sim.simulate_prefetch(4096, 32, dtype_bytes=0.5)

print("=== 权重预取策略分析 (7B INT4, 高通Hexagon) ===")
print(f"单层权重: {prefetch_result['layer_weight_mb']:.1f} MB")
print(f"单层预取时间: {prefetch_result['prefetch_time_us']:.1f} μs")
print(f"单层计算时间: {prefetch_result['compute_time_us']:.1f} μs")
print(f"预取是否完全隐藏: {'是' if prefetch_result['prefetch_fully_hidden'] else '否'}")
print(f"无预取总延迟: {prefetch_result['no_prefetch_total_ms']:.2f} ms")
print(f"有预取总延迟: {prefetch_result['prefetch_total_ms']:.2f} ms")
print(f"加速比: {prefetch_result['speedup']:.2f}x")

stream_result = sim.simulate_streaming(4096, 32, available_dram_mb=2048, dtype_bytes=0.5)
print(f"\n=== 流式加载策略分析 (7B INT4, DRAM=2GB) ===")
print(f"模型总权重: {stream_result['total_weight_mb']:.0f} MB")
print(f"可用DRAM: {stream_result['available_dram_mb']} MB")
print(f"可容纳层数: {stream_result['layers_fit']}/{32}")
print(f"需要分块数: {stream_result['n_chunks']}")
print(f"每块Flash加载时间: {stream_result['flash_load_time_per_chunk_ms']:.1f} ms")
print(f"总Flash加载时间: {stream_result['total_flash_load_ms']:.1f} ms")

print(f"\n=== 产业实践要点 ===")
print(f"1. 权重预取是NPU推理的标准优化，SDK通常内置")
print(f"2. Decode阶段计算时间<预取时间，预取只能部分隐藏延迟")
print(f"3. 流式加载适用于大模型(70B+)在内存受限设备上部署")
print(f"4. Flash带宽(2-5 GB/s)远低于DRAM(60+ GB/s)，流式加载延迟显著")
print(f"5. 混合策略: 热层(Attention)常驻DRAM，冷层(FFN)按需流式加载")

---
## 5.3.8 推测解码与端侧加速策略

推测解码（Speculative Decoding）是端侧LLM推理最重要的加速技术之一。它通过小模型预测+大模型验证的方式，在不损失精度的前提下将decode吞吐提升2-3x，特别适合memory-bound的端侧场景。

### 推测解码的核心思想

Decode阶段的瓶颈是内存带宽：每生成1个token需要读取全部权重，但MAC利用率极低（<5%）。推测解码利用这些空闲的MAC算力：

```
标准Decode:  生成1 token → 读取全部权重 → MAC利用率3%
             ████████████████████████████░░░░░░░░░░░░░░░░░░░░

推测Decode:  小模型生成K tokens → 大模型并行验证K tokens
             ████████████████████████████████████████████████
             → 平均接受率α → 有效加速 = 1/(1-α) ≈ 2-3x
```

### 推测解码的数学分析

设小模型（draft model）每token生成时间为$t_d$，大模型（target model）每token验证时间为$t_t$，接受率为$\alpha$：

$$\text{加速比} = \frac{t_t}{\alpha \cdot t_d + (1 - \alpha) \cdot t_t + t_t / K}$$

当$\alpha > 0.7$且$t_d \ll t_t$时，加速比可达2-3x。

### 端侧推测解码的特殊优势

| 优势 | 说明 |
|------|------|
| **MAC利用率提升** | 大模型验证K个token时，batch=K的GEMM比batch=1效率高K倍 |
| **无需额外内存** | 小模型可常驻内存（仅100-300MB），权重无需反复加载 |
| **精度无损** | 大模型验证保证输出分布与标准采样完全一致 |
| **延迟不增加** | 最坏情况（全部拒绝）延迟仅增加$t_d$，通常<10% |

### 端侧推测解码的实践方案

| 方案 | Draft模型 | 适用场景 | 预期加速 |
|------|----------|---------|----------|
| **同系小模型** | Qwen2.5-0.5B → Qwen2.5-7B | 同架构模型，接受率高 | 2-3x |
| **自推测** | 7B模型前N层 → 完整7B | 无需额外模型，内存友好 | 1.5-2x |
| **N-gram推测** | 基于历史token的n-gram表 | 无需模型，零额外内存 | 1.3-1.5x |
| **Early Exit** | 中间层提前输出 → 后续层验证 | 模型内部机制，无额外开销 | 1.5-2x |

### Continuous Batching（连续批处理）

端侧多用户场景下，continuous batching通过iteration-level调度提升吞吐：

- **传统批处理**：等所有请求完成同一阶段后再处理下一批 → 有气泡
- **连续批处理**：每个iteration结束后，完成的请求离开，新请求加入 → 无气泡
- **端侧适用性**：智能音箱/车载场景常有多个并发请求，continuous batching可提升2-4x吞吐

In [ ]:
class SpeculativeDecodingSimulator:
    """推测解码性能模拟器"""
    def __init__(self, target_tps=10, draft_tps=50):
        self.target_tps = target_tps
        self.draft_tps = draft_tps

    def simulate_speculative(self, n_tokens, k_spec=5, acceptance_rate=0.7):
        """模拟推测解码过程"""
        target_time_per_token = 1000 / self.target_tps
        draft_time_per_token = 1000 / self.draft_tps

        total_time = 0
        tokens_generated = 0
        steps = 0
        while tokens_generated < n_tokens:
            draft_time = k_spec * draft_time_per_token
            verify_time = target_time_per_token
            step_time = draft_time + verify_time
            accepted = min(int(k_spec * acceptance_rate), k_spec - 1)
            if accepted == 0:
                accepted = 1
            tokens_generated += accepted
            total_time += step_time
            steps += 1

        effective_tps = tokens_generated / total_time * 1000
        baseline_time = n_tokens * target_time_per_token
        speedup = baseline_time / total_time
        return {
            'tokens': tokens_generated,
            'steps': steps,
            'total_time_ms': total_time,
            'effective_tps': effective_tps,
            'speedup': speedup,
            'acceptance_rate': acceptance_rate,
            'k_spec': k_spec,
        }

sim = SpeculativeDecodingSimulator(target_tps=10, draft_tps=50)

print("=== 推测解码性能模拟 ===")
print(f"目标模型: 10 tok/s, Draft模型: 50 tok/s")
print(f"\n{'K(推测长度)':<12} {'接受率':<10} {'有效tok/s':<12} {'加速比':<10} {'说明'}")
print("-" * 65)
for k in [3, 5, 7, 10]:
    for alpha in [0.6, 0.7, 0.8, 0.9]:
        result = sim.simulate_speculative(100, k_spec=k, acceptance_rate=alpha)
        note = ''
        if k == 5 and alpha == 0.7:
            note = '← 典型配置'
        if k == 5 and alpha == 0.8:
            note = '← 同系小模型'
        print(f"{k:<12} {alpha:<10.1f} {result['effective_tps']:<12.1f} {result['speedup']:<10.2f}x {note}")

print(f"\n=== Continuous Batching吞吐模拟 ===")
class ContinuousBatchingSimulator:
    def __init__(self, decode_tps=10, max_batch=4):
        self.decode_tps = decode_tps
        self.max_batch = max_batch

    def simulate(self, n_requests, avg_tokens, request_rate_per_s):
        batch_utilization = min(n_requests * request_rate_per_s * avg_tokens / self.decode_tps, self.max_batch) / self.max_batch
        effective_throughput = self.decode_tps * min(n_requests, self.max_batch) * 0.85
        per_user_tps = effective_throughput / max(n_requests, 1)
        return {'throughput': effective_throughput, 'per_user_tps': per_user_tps, 'batch_util': batch_utilization}

cb_sim = ContinuousBatchingSimulator(decode_tps=10, max_batch=4)
print(f"单请求吞吐: 10 tok/s, 最大batch: 4")
for n_req in [1, 2, 3, 4]:
    r = cb_sim.simulate(n_req, avg_tokens=50, request_rate_per_s=0.5)
    print(f"  {n_req}并发请求: 总吞吐={r['throughput']:.0f} tok/s, 每用户={r['per_user_tps']:.1f} tok/s")

print(f"\n=== 端侧加速策略选择指南 ===")
print(f"1. 单用户+低延迟需求 → 推测解码 (2-3x加速, 精度无损)")
print(f"2. 多用户+高吞吐需求 → Continuous Batching (2-4x吞吐提升)")
print(f"3. 内存受限 → N-gram推测或自推测 (零额外内存)")
print(f"4. 追求极致性能 → 推测解码+Continuous Batching组合")
print(f"5. 关键约束: 推测解码需要draft模型与target模型输出分布接近")

---
## 5.3.9 总结与最佳实践

### 硬件感知优化的核心原则

1. **Roofline驱动**：先用Roofline分析定位瓶颈（compute-bound vs memory-bound），再选择优化方向
2. **量化是第一优化**：对memory-bound的decode阶段，量化直接减少数据搬运量，效果最显著
3. **Tiling是基础**：SRAM容量有限，Tiling是所有NPU优化的基础
4. **双缓冲隐藏延迟**：计算与搬运重叠，在compute-bound场景效果最好
5. **功耗是硬约束**：端侧推理必须在TDP内完成，热管理是长期运行的保障

### 优化策略选择矩阵

| 瓶颈类型 | Prefill (compute-bound) | Decode (memory-bound) |
|---------|----------------------|---------------------|
| **首选优化** | 算子融合 / Flash Attention | 量化 (W4A16) |
| **次选优化** | 增大batch / 张量并行 | KV量化 / 权重预取 |
| **进阶优化** | 硬件感知NAS | 双缓冲 / 算子融合 |
| **量化效果** | 有限 (~1.2x) | 显著 (~3-4x) |

### 硬件感知优化Checklist

- [ ] Roofline分析：确定各算子的瓶颈类型
- [ ] 量化配置：根据瓶颈类型选择W4A16/W8A8
- [ ] Tiling规划：确保所有算子fit进SRAM
- [ ] 双缓冲启用：确认NPU SDK已启用计算与搬运重叠
- [ ] 算子融合：减少中间结果DRAM写回
- [ ] 功耗测试：长时间推理的温度和性能稳定性
- [ ] 降级策略：热节流时的自动降级方案
- [ ] Profile验证：优化后重新Profile，确认瓶颈已消除